# **Anime Recommendation System using Cosine Similarity**



## 1. Data Preprocessing

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load dataset
df = pd.read_csv('/content/anime.csv')
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [5]:
# Check missing values
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [6]:
# Handle missing values
df['genre'] = df['genre'].fillna('NaN')
df['rating'] = df['rating'].fillna(df['rating'].mean())
df['members'] = df['members'].fillna(df['members'].median())

In [7]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,25
episodes,0
rating,0
members,0


In [13]:
df.type

,type
0,Movie
1,TV
2,TV
3,TV
4,TV
...,...
12289,OVA
12290,OVA
12291,OVA
12292,OVA


## 2. Feature Extraction

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

In [15]:
# TF-IDF for genres
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(df['genre'])

In [22]:
tfidf

TfidfVectorizer(stop_words='english')

In [23]:
genre_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 40480 stored elements and shape (12294, 47)>

In [16]:
# Normalize numerical features
scaler = MinMaxScaler()
num_features = scaler.fit_transform(df[['rating', 'members']])

In [21]:
scaler

MinMaxScaler()

In [20]:
num_features

array([[9.24369748e-01, 1.97872202e-01],
       [9.11164466e-01, 7.82770102e-01],
       [9.09963986e-01, 1.12689267e-01],
       ...,
       [3.85354142e-01, 2.11063682e-04],
       [3.97358944e-01, 1.67667411e-04],
       [4.54981993e-01, 1.35120208e-04]])

In [18]:
# Combine features
from scipy.sparse import hstack

In [19]:
feature_matrix = hstack([genre_matrix, num_features])
feature_matrix

<COOrdinate sparse matrix of dtype 'float64'
	with 65066 stored elements and shape (12294, 49)>

## 3. Recommendation System

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)

In [26]:
cosine_sim

array([[1.        , 0.53235245, 0.46247873, ..., 0.24157166, 0.24807781,
        0.27820617],
       [0.53235245, 1.        , 0.51682949, ..., 0.20971947, 0.2153503 ,
        0.24148453],
       [0.46247873, 0.51682949, 1.        , ..., 0.24118659, 0.24768518,
        0.27776899],
       ...,
       [0.24157166, 0.20971947, 0.24118659, ..., 1.        , 0.99994581,
        0.99824985],
       [0.24807781, 0.2153503 , 0.24768518, ..., 0.99994581, 1.        ,
        0.99881138],
       [0.27820617, 0.24148453, 0.27776899, ..., 0.99824985, 0.99881138,
        1.        ]])

In [27]:
def recommend_anime(title, top_n=5, threshold=0.3):
    if title not in df['name'].values:
        return 'Anime not found in dataset.'

    idx = df[df['name'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    recommendations = []
    for i, score in sim_scores[1:]:
        if score >= threshold:
            recommendations.append((df.iloc[i]['name'], score))
        if len(recommendations) >= top_n:
            break

    return recommendations

In [29]:
# Example recommendation
recommend_anime(df['name'].iloc[0])

[('Wind: A Breath of Heart OVA', np.float64(0.9628328816787185)),
 ('Wind: A Breath of Heart (TV)', np.float64(0.9589029479221463)),
 ('Aura: Maryuuin Kouga Saigo no Tatakai', np.float64(0.958359297252002)),
 ('Shakugan no Shana II (Second)', np.float64(0.9178070215979311)),
 ('Angel Beats!: Another Epilogue', np.float64(0.9161215438910484))]

## 4. Analysis & Improvements
- Better genre encoding (word embeddings)
- Include user interaction data
- Tune similarity thresholds dynamically

## Interview Questions
**1. Difference between user-based and item-based collaborative filtering?**

- User-based: recommends items liked by similar users.
- Item-based: recommends items similar to those a user liked.

**2. What is collaborative filtering?**

Collaborative filtering uses user-item interactions to predict preferences based on similarities among users or items.